# FlashAttention in JAX Pallas — Colab Runner

Runs the three-way attention comparison (eager JAX vs. `jax.jit`/XLA vs. hand-written Pallas kernel).

**Pick a runtime first** (Runtime → Change runtime type):

| Runtime | Cost | What you get |
|---|---|---|
| CPU | free | correctness validation only (Pallas interpret mode — slow, timings meaningless) |
| T4 GPU | free tier | real benchmark via the Triton backend |
| L4 GPU / TPU | Colab Pro compute units | the numbers for the README |

Workflow: validate on CPU/T4 first, only then spend compute units on L4/TPU.

In [ ]:
# Clone and install. Edit REPO_URL after you push to GitHub.
REPO_URL = "https://github.com/YOUR_USERNAME/pallas-flash-attention.git"

import os
if not os.path.exists("pallas-flash-attention"):
    !git clone {REPO_URL}
%cd pallas-flash-attention
%pip install -q -e . pytest matplotlib

In [ ]:
# Check what we're running on. Colab GPU runtimes ship JAX with CUDA support;
# on a TPU runtime, if the backend below says 'cpu', run:
#   %pip install -q -U "jax[tpu]"   # then restart the runtime
import jax
print("JAX version:", jax.__version__)
print("Backend:    ", jax.default_backend())
print("Devices:    ", jax.devices())

In [ ]:
# Step 1 — correctness gate. Must be green before any benchmarking.
!python -m pytest tests -q

In [ ]:
# Step 2 — benchmark sweep.
import jax
if jax.default_backend() == "cpu":
    # interpret mode: tiny sizes, plumbing check only
    !python benchmarks/run_benchmark.py --seq-lens 256 512 --dtype f32 --iters 3 --output results/results.csv
else:
    !python benchmarks/run_benchmark.py --seq-lens 512 1024 2048 4096 8192 --dtype bf16 --output results/results.csv

In [ ]:
# Step 3 — plots. Set --device to match the runtime:
#   L4-bf16-tensor | T4-fp16-tensor | T4-fp32 | TPUv5e-bf16
!python benchmarks/plot_results.py results/results.csv --device L4-bf16-tensor

from IPython.display import Image, display
for name in ["runtime_vs_seqlen.png", "throughput_vs_seqlen.png", "roofline.png"]:
    display(Image(f"results/{name}"))

## Interpreting the results

- **naive vs. XLA**: how much automatic fusion buys you for free.
- **XLA vs. Pallas**: the payoff of manual tiling + online softmax. The gap should *grow* with sequence length — the naive/XLA paths pay O(N²) HBM traffic, the Pallas kernel pays O(N·d).
- **Roofline**: naive/XLA points sit left of the ridge (memory-bound); the Pallas kernel's higher arithmetic intensity moves it right, toward compute-bound.
- Expect naive to OOM at long sequence lengths — that failure *is* a data point (the memory cliff).

Copy `results/*.png` and the CSV into the repo, fill in the README results table, commit.